# Section 4.5 — Corrupted Initialization: Binary Walking Detection

Three arms, same data, same tuning. Only the initial parameters and
whether learning is switched on differ.

| Arm | init | learning |
|---|---|---|
| Naive-frozen | c = 1, σ² = 1 | off |
| Sabotaged-frozen | c = 0.001, σ² = 100 | off |
| AKF-recovery | c = 0.001, σ² = 100 | on |

---

### The problem with the previous version, and the fix

The cumulative estimator computes

```
c_hat(k) = mean( y[0..k] / T[0..k] )
```

over the **whole history**. At k = 0 the initial value is overwritten
outright; by k = 1 it carries no weight at all. So the sabotage survived
exactly one sample. The parameter plots started at 3.95 and 2.3, never
anywhere near 0.001, and naive-frozen and AKF-recovery both finished at
1.35% — because there was nothing to recover *from*.

**`K0` fixes this.** Learning is held off until k ≥ K0, so the sabotaged
parameters actually drive the filter for K0 samples before the estimator
is allowed to move them. `K0 = 0` reproduces the old (uninformative)
behaviour; `K0 = 200` gives the filter a genuine bad start to recover
from.

This also connects the experiment to the burn-in discussion: K0 is the
same quantity, used deliberately rather than incidentally.

### Also removed

The supervising-trajectory table and the person-counting script had been
pasted into this notebook. Both belong to Section 4 and are now in
`Chapter4_Supervising_Trajectory.ipynb`.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
sns.set_palette("deep")

CSV = "../data/binary_walking_1851.csv"

# shared tuning, identical across all three arms
SIG_E2, C_E = 1.0, 1.0
Q_VAL, P0_VAL, X0_VAL = 10.0, 1.0, 0.0
EPS = 1e-6

# ---- the bad start ----
# A coupling three orders of magnitude too small paired with a noise
# variance two orders too large. Both drive the Kalman gain for the
# inexpert channels towards zero, so the fused estimate begins driven
# by the camera alone.
SABOTAGE_C = -3
SABOTAGE_S = 3

# ---- how long the bad start is allowed to persist ----
# Learning is held off until k >= K0. With K0 = 0 the sabotage is
# overwritten at the first sample and the experiment measures nothing.
K0 = 0

TEACHER_MODE = "gt"      # "gt", "expert", or "fused"

C_REF = {1: 4.332, 2: 6.360}


## 1. Load and build ground truth

Identical to the Section 4 notebook — Eq. (3.9), with adjacent presses
de-duplicated.


In [2]:
df = pd.read_csv(CSV)
df = df[["human", "mouse", "dBFS_mic1", "dBFS_mic2"]].dropna().reset_index(drop=True)

m = df["mouse"].astype(int)
m = m.mask((m == 1) & (m.shift(1, fill_value=0) == 1), 0)
walking_gt = (m.cumsum() % 2).astype(int)

N  = len(df)
X  = np.where(walking_gt.to_numpy() == 1, 1, -1)
y_e = np.where(df["human"].astype(int).to_numpy() == 1, 1.0, -1.0)

y1 = df["dBFS_mic1"].astype(float).to_numpy(); y1 = y1 - y1.mean()
y2 = df["dBFS_mic2"].astype(float).to_numpy(); y2 = y2 - y2.mean()

n_neg, n_pos = int((X == -1).sum()), int((X == 1).sum())
print(f"N = {N}   class counts = {n_neg} / {n_pos}")
assert N == 1851 and (n_neg, n_pos) == (1168, 683), "does not match Table 1"


N = 1851   class counts = 1168 / 683


## 2. The filter

`k0` is the new argument. Everything else matches the Section 4
notebook.


In [3]:
def run(c_init, s_init, learn, teacher_mode="fused", k0=0):
    """Expert + two inexpert sensors.

    learn=False : parameters frozen at (c_init, s_init) throughout.
    learn=True  : frozen until k >= k0, then updated online.
    """
    A = np.eye(1)
    Q = Q_VAL * np.eye(1)

    Xf, Pf = np.array([X0_VAL]), np.eye(1) * P0_VAL
    Xe, Pe = np.array([X0_VAL]), np.eye(1) * P0_VAL

    c1h = c2h = float(c_init)
    s1h = s2h = float(s_init)

    dec, exp_dec = np.zeros(N), np.zeros(N)
    hist = {k: np.zeros(N) for k in ("c1", "c2", "s1", "s2")}

    C_e, R_e = np.array([[C_E]]), np.array([[SIG_E2]])

    for k in range(N):
        # expert-only filter
        Xm = A @ Xe.reshape(-1, 1)
        Pm = A @ Pe @ A.T + Q
        K  = Pm @ C_e.T @ np.linalg.inv(C_e @ Pm @ C_e.T + R_e)
        Xe = (Xm + K @ (y_e[k] - C_e @ Xm)).flatten()
        Pe = (np.eye(1) - K @ C_e) @ Pm
        exp_dec[k] = -1 if Xe[0] < 0 else 1

        # fused filter
        C = np.array([[C_E], [c1h], [c2h]])
        R = np.diag([SIG_E2, max(s1h, EPS), max(s2h, EPS)])
        Xm = A @ Xf.reshape(-1, 1)
        Pm = A @ Pf @ A.T + Q
        resid = np.array([y_e[k], y1[k], y2[k]]) - (C @ Xm).flatten()
        K  = Pm @ C.T @ np.linalg.inv(C @ Pm @ C.T + R)
        Xf = (Xm + K @ resid.reshape(-1, 1)).flatten()
        Pf = (np.eye(1) - K @ C) @ Pm
        dec[k] = -1 if Xf[0] < 0 else 1

        # parameter update -- held off until k >= k0
        if learn and k >= k0:
            if   teacher_mode == "gt":     T = X[:k+1].astype(float)
            elif teacher_mode == "expert": T = exp_dec[:k+1]
            elif teacher_mode == "fused":  T = dec[:k+1]
            else: raise ValueError(teacher_mode)

            denom = T.copy()
            denom[np.abs(denom) < EPS] = EPS
            c1h = float(np.mean(y1[:k+1] / denom))
            s1h = float(np.var(y1[:k+1] - c1h * T))
            c2h = float(np.mean(y2[:k+1] / denom))
            s2h = float(np.var(y2[:k+1] - c2h * T))

        for key, val in zip(("c1", "c2", "s1", "s2"), (c1h, c2h, s1h, s2h)):
            hist[key][k] = val

    return dec, hist


def cum_err(dec):
    return 100 * np.cumsum(dec != X) / (np.arange(N) + 1)


## 3. Run the three arms


In [4]:
naive_dec, naive_h = run(1.0, 1.0, learn=False)
sab_dec,   sab_h   = run(SABOTAGE_C, SABOTAGE_S, learn=False)
akf_dec,   akf_h   = run(SABOTAGE_C, SABOTAGE_S, learn=True,
                         teacher_mode=TEACHER_MODE, k0=K0)

cum_naive, cum_sab, cum_akf = cum_err(naive_dec), cum_err(sab_dec), cum_err(akf_dec)

print(f"teacher = {TEACHER_MODE}   K0 = {K0}\n")
print(f"  Naive-frozen     (c=1, s=1, frozen)          : {cum_naive[-1]:6.2f}%")
print(f"  Sabotaged-frozen (c={SABOTAGE_C}, s={SABOTAGE_S:.0f}, frozen) : {cum_sab[-1]:6.2f}%")
print(f"  AKF-recovery     (same init, learning on)    : {cum_akf[-1]:6.2f}%")

print(f"\nAKF final parameters:")
print(f"  c1 = {akf_h['c1'][-1]:8.3f}  (ref {C_REF[1]:.3f})")
print(f"  c2 = {akf_h['c2'][-1]:8.3f}  (ref {C_REF[2]:.3f})")

print(f"\nAKF parameters at k = K0-1 (just before learning starts):")
print(f"  c1 = {akf_h['c1'][max(K0-1,0)]:8.3f}   c2 = {akf_h['c2'][max(K0-1,0)]:8.3f}")
print("  ^^ these should still be the sabotaged value. If they are not, "
      "K0 is not doing its job.")


teacher = gt   K0 = 0

  Naive-frozen     (c=1, s=1, frozen)          :   1.35%
  Sabotaged-frozen (c=-3, s=3, frozen) :  99.08%
  AKF-recovery     (same init, learning on)    :   1.46%

AKF final parameters:
  c1 =    4.035  (ref 4.332)
  c2 =    5.923  (ref 6.360)

AKF parameters at k = K0-1 (just before learning starts):
  c1 =    3.927   c2 =    2.294
  ^^ these should still be the sabotaged value. If they are not, K0 is not doing its job.


## 4. Does the learning actually do anything?

Aggregate error is dominated by samples where the camera is already
right and the microphones cannot change the outcome. The rescue rate
restricts attention to the samples where the camera is wrong — the only
regime in which microphone calibration can matter.

If naive-frozen and sabotaged-frozen rescue the same fraction, the
microphones are contributing nothing and the process-noise dynamics are
doing the work. For the learning to matter, AKF-recovery must rescue
more than sabotaged-frozen.


In [5]:
expert_alone = np.where(y_e < 0, -1, 1)
wrong = (expert_alone != X)
n_wrong = int(wrong.sum())

print(f"camera wrong on {n_wrong}/{N} samples ({100*n_wrong/N:.2f}%)\n")
for label, dec in (("Naive-frozen", naive_dec),
                   ("Sabotaged-frozen", sab_dec),
                   ("AKF-recovery", akf_dec)):
    r = 100 * ((dec == X) & wrong).sum() / n_wrong
    print(f"  {label:<18} rescue rate {r:6.2f}%")

n_diff = int((naive_dec != akf_dec).sum())
print(f"\nnaive vs AKF differ at {n_diff}/{N} samples")
if n_diff == 0:
    print("  ^^ identical classifications despite different parameters. "
          "The camera is dominating; this run says nothing about "
          "calibration quality.")


camera wrong on 527/1851 samples (28.47%)

  Naive-frozen       rescue rate  96.39%
  Sabotaged-frozen   rescue rate   1.90%
  AKF-recovery       rescue rate  95.83%

naive vs AKF differ at 14/1851 samples


## 5. Recovery point

The first index after which the AKF error stays permanently below half
the sabotaged-frozen final error.


In [6]:
thr = 0.5 * cum_sab[-1]
below = cum_akf <= thr
rec = None
for i in range(K0, N):
    if below[i] and below[i:].all():
        rec = i
        break

if rec is not None:
    print(f"AKF error drops permanently below {thr:.2f}% at k = {rec} "
          f"({rec - K0} samples after learning starts)")
else:
    print(f"AKF error never settles permanently below {thr:.2f}%")

gap = cum_sab - cum_akf
g = int(np.argmax(gap))
print(f"largest gap: {gap[g]:.2f} points at k = {g} "
      f"(AKF {cum_akf[g]:.2f}%, sabotaged {cum_sab[g]:.2f}%)")


AKF error drops permanently below 49.54% at k = 2 (2 samples after learning starts)
largest gap: 97.70 points at k = 1783 (AKF 1.40%, sabotaged 99.10%)


## 6. Figures


In [7]:
# ---- Figure: the three curves ----
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(cum_naive, ls="--", color="gray", lw=1.6,
        label=f"Naive-frozen ({cum_naive[-1]:.2f}%)")
ax.plot(cum_sab, ls=":", color="firebrick", lw=1.6,
        label=f"Sabotaged-frozen ({cum_sab[-1]:.2f}%)")
ax.plot(cum_akf, color="steelblue", lw=2,
        label=f"AKF-recovery ({cum_akf[-1]:.2f}%)")
if K0 > 0:
    ax.axvline(K0, color="black", ls="-.", lw=1, alpha=0.6)
    ax.text(K0, ax.get_ylim()[1] * 0.92, f"  learning starts, $K_0$ = {K0}",
            fontsize=8)
ax.set_xlabel("Time step $k$")
ax.set_ylabel("Cumulative classification error [%]")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig("badstart_error_binary.pdf")
plt.show()

# ---- Figure: parameter recovery ----
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for idx, (ax, key) in enumerate(zip(axes, ("c1", "c2")), start=1):
    ax.axhline(C_REF[idx], color="black", ls="--", lw=1.4,
               label=f"Offline reference $c_{idx}^*$ = {C_REF[idx]:.3f}")
    ax.axhline(SABOTAGE_C, color="firebrick", ls=":", lw=1.4,
               label=f"Sabotaged start ({SABOTAGE_C})")
    ax.plot(akf_h[key], color="steelblue", lw=1.8,
            label=f"AKF-recovery ({akf_h[key][-1]:.3f})")
    if K0 > 0:
        ax.axvline(K0, color="black", ls="-.", lw=1, alpha=0.6)
    ax.set_ylabel(rf"$\hat{{c}}_{idx}(k)$")
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8)
axes[1].set_xlabel("Time step $k$")
fig.tight_layout()
fig.savefig("badstart_couplings_binary.pdf")
plt.show()

print("saved: badstart_error_binary.pdf, badstart_couplings_binary.pdf")


/tmp/ipykernel_1051/3594696960.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved: badstart_error_binary.pdf, badstart_couplings_binary.pdf


/tmp/ipykernel_1051/3594696960.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ---- Figure: recovery vs sign lock-in from the inverted start ----
INVERT_C, INVERT_S = -3.0, 3.0

gt_dec,  gt_h  = run(INVERT_C, INVERT_S, learn=True, teacher_mode="gt",    k0=0)
fus_dec, fus_h = run(INVERT_C, INVERT_S, learn=True, teacher_mode="fused", k0=0)

print(f"annotated   final error {100*(gt_dec  != X).mean():.2f}%"
      f"   c1 {gt_h['c1'][-1]:.3f}  c2 {gt_h['c2'][-1]:.3f}")
print(f"self-taught final error {100*(fus_dec != X).mean():.2f}%"
      f"   c1 {fus_h['c1'][-1]:.3f}  c2 {fus_h['c2'][-1]:.3f}")

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for idx, (ax, key) in enumerate(zip(axes, ("c1", "c2")), start=1):
    ax.axhline(0, color="gray", lw=0.8, alpha=0.6)
    ax.axhline(C_REF[idx], color="black", ls="--", lw=1.4,
               label=rf"Offline reference $c_{idx}^*$ = {C_REF[idx]:.3f}")
    ax.axhline(INVERT_C, color="firebrick", ls=":", lw=1.4,
               label=f"Inverted start ({INVERT_C:.0f})")
    ax.plot(gt_h[key],  color="seagreen",  lw=1.8,
            label=f"Annotated ({gt_h[key][-1]:.3f})")
    ax.plot(fus_h[key], color="steelblue", lw=1.8,
            label=f"Self-taught ({fus_h[key][-1]:.3f})")
    ax.set_ylabel(rf"$\hat{{c}}_{idx}(k)$")
    ax.grid(alpha=0.3)
    ax.legend(loc="center right", fontsize=8)
axes[1].set_xlabel("Time step $k$")
fig.tight_layout()
fig.savefig("corrupted_couplings_binary.pdf")
plt.show()

annotated   final error 1.46%   c1 4.035  c2 5.923
self-taught final error 99.03%   c1 -4.079  c2 -5.983


/tmp/ipykernel_1051/260353368.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. K0 sweep

How long a bad start can the estimator absorb? This is the experiment
the chapter actually wants: the sabotage has to persist for a while
before recovery means anything.


In [9]:
print(f"{'K0':>6}{'final error':>14}{'c1':>10}{'c2':>10}")
print("-" * 40)
for k0 in (0, 50, 100, 200, 400, 800):
    d, h = run(SABOTAGE_C, SABOTAGE_S, learn=True,
               teacher_mode=TEACHER_MODE, k0=k0)
    print(f"{k0:>6}{100*(d != X).mean():>13.2f}%"
          f"{h['c1'][-1]:>10.3f}{h['c2'][-1]:>10.3f}")
print("-" * 40)
print(f"frozen (no learning at all): {cum_sab[-1]:.2f}%")


    K0   final error        c1        c2
----------------------------------------


     0         1.46%     4.035     5.923


    50         4.05%     4.035     5.923


   100         6.48%     4.035     5.923


   200        11.67%     4.035     5.923


   400        22.20%     4.035     5.923


   800        43.22%     4.035     5.923
----------------------------------------
frozen (no learning at all): 99.08%
